# Course notebook — Agent Systems

Design agentic systems as bounded stateful architectures rather than treating autonomy as a model feature.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Workflows and agent loops


### Lecture 01 · Agent or Workflow?

Define agents by goal-directed action selection in a loop and contrast them with deterministic workflows and single tool calls.

**Predict before running.** Use the least autonomy that handles the uncertainty in the task. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Model the same task as a workflow and as an agent, then compare.


In [ ]:
"""Lab 8.1: Agent or Workflow?"""

WORKFLOW = ["create_account", "assign_laptop", "grant_access"]
AGENT = {"goal": "complete onboarding", "actions": WORKFLOW, "replans": True}
def run(steps, fail_at=None):
    done = []
    for i, s in enumerate(steps):
        if fail_at == i:
            return done, "paused"
        done.append(s)
    return done, "complete"
print("workflow:", run(WORKFLOW, fail_at=1))
print("agent can resume:", AGENT["replans"])


### Lecture 02 · The Agent Loop

Connect goal, state, planning, action, observation, reflection, and termination into a bounded state machine.

**Predict before running.** An agent loop without explicit state and stopping rules is an unreliable retry loop. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Extend the included agent loop with failures and checkpointing.


In [ ]:
"""Lab 8.2: The Agent Loop"""

CHAPTER = "8.2"
print("chapter hook:", CHAPTER)
state = {"step": 0, "observations": [], "done": False, "budget": 3}
while not state["done"] and state["step"] < state["budget"]:
    state["step"] += 1
    obs = f"obs-{state['step']}"
    state["observations"].append(obs)
    state["done"] = state["step"] == 3
print(state)
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Identify where a fixed workflow is sufficient. Write an invariant for a bounded loop.


## Recovery and patterns


### Lecture 03 · Agent Memory and Recovery

Manage working state, episodic history, durable checkpoints, resumability, compensation, and idempotent tools.

**Predict before running.** Continuity requires durable state and recoverable effects, not merely longer context. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Persist and resume an interrupted multi-step run.


In [ ]:
"""Lab 8.3: Agent Memory and Recovery"""

checkpoints = []
state = {"step": "assign_laptop", "order_id": None}
def save(state):
    checkpoints.append(dict(state))
def resume():
    return checkpoints[-1] if checkpoints else None
save({"step": "create_account", "done": True})
state = resume()
print("resume_at:", state)


### Lecture 04 · Agent Patterns

Apply planner–executor, supervisor–worker, reviewer, evaluator–optimizer, routing, and human-approval patterns.

**Predict before running.** Patterns trade flexibility for additional state, calls, latency, and failure surfaces. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement two patterns and measure coordination overhead.


In [ ]:
"""Lab 8.4: Agent Patterns"""

patterns = {
    "monolith": {"calls": 1, "latency": 1.0},
    "planner_executor": {"calls": 3, "latency": 1.6},
    "supervisor_worker": {"calls": 5, "latency": 2.1},
}
task_risk = "high"
choice = "supervisor_worker" if task_risk == "high" else "planner_executor"
print({"pattern": choice, **patterns[choice]})


**Section checkpoint.** Record what must survive a restart so an approved action is handled once.


## Coordination and operations


### Lecture 05 · Multi-Agent Systems

Study delegation, role boundaries, communication, shared state, consensus, conflict, security, and why many tasks do not need multiple agents.

**Predict before running.** More agents increase organizational complexity faster than raw capability. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Split a research task across workers and compare with one-agent parallel tools.


In [ ]:
"""Lab 8.5: Multi-Agent Systems"""

CHAPTER = "8.5"
print("chapter hook:", CHAPTER)
workers = {"A": "fetch HR policy", "B": "fetch IT policy"}
shared = []
for w, task in workers.items():
    shared.append({"worker": w, "result": f"evidence from {task}"})
conflicts = len({r["result"][:10] for r in shared}) < len(shared)
print({"evidence": shared, "conflict": conflicts})
print("---")
print("change one input above, predict output, re-run")


### Lecture 06 · Operating Long-Running Agents

Design durable orchestration, queues, scheduling, leases, approvals, monitoring, incident response, and safe cancellation.

**Predict before running.** Long-running agents are distributed systems with probabilistic decision components. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create an SLO and runbook for a day-long agent workflow.


In [ ]:
"""Lab 8.6: Operating Long-Running Agents"""

CHAPTER = "8.6"
print("chapter hook:", CHAPTER)
SLO_HOURS = 24
lease_minutes = 30
elapsed = 12 * 60
renewals = elapsed // lease_minutes
print({"elapsed_min": elapsed, "lease_renewals": renewals, "within_slo": elapsed <= SLO_HOURS * 60})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Choose a stopping rule and specify what evidence an operator receives on failure.


## Course workshop

Trace the example from planning to its approval pause, then resume after approval and verify that it reaches completion.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Bounded agent state machine with checkpoint, approval, and termination."""
from dataclasses import dataclass, asdict
import json


@dataclass
class State:
    step: str = "plan"
    attempts: int = 0
    approved: bool = False
    done: bool = False


def transition(state):
    if state.attempts >= 5: return State("stopped", state.attempts, state.approved, True)
    if state.step == "plan": return State("draft", state.attempts + 1)
    if state.step == "draft": return State("approval", state.attempts + 1)
    if state.step == "approval" and not state.approved: return State("approval", state.attempts + 1)
    return State("complete", state.attempts + 1, state.approved, True)


state = State()
while not state.done:
    print(json.dumps(asdict(state)))
    if state.step == "approval": state.approved = True
    state = transition(state)
print(json.dumps(asdict(state)))


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
state = State()
state = transition(transition(state))
assert state.step == "approval"
state = transition(state)
assert state.step == "approval"
state.approved = True
state = transition(state)
assert state.step == "complete" and state.done
print('Workshop checks passed')


## Course project

Build a durable multi-step agent with checkpoints, approval gates, evaluation, and observable termination.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
